In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

from latex_table import linear_regression

In [2]:
# Directories
PROJECT = "C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"
DATA = os.path.join(PROJECT, "Data")
FIGURES = os.path.join(PROJECT, "Figures")

In [3]:
# Load results from trading_daily notebook
results_file = os.path.join(DATA, 'trading_daily_results.pkl')
with open(results_file, 'rb') as f:
    results = pickle.load(f)

portfolios = results['portfolios']
portfolios_filtered = results['portfolios_filtered']
ff_factors = results['ff_factors']
prediction_columns = results['prediction_columns']
MODELS = results['MODELS']

print(f"Loaded results with {len(prediction_columns)} models")
print(f"Models: {list(MODELS.keys())}")

Loaded results with 9 models
Models: ['lr', 'lasso', 'elasticnet', 'nn', 'nn_tuned_1layer', 'lr_all', 'lasso_all', 'elasticnet_all', 'nn_all']


# Plot log cumulative returns

In [ ]:
# Group models for plotting
model_groups = {
    'Two Features': ['lr', 'lasso', 'elasticnet', 'nn', 'nn_tuned_1layer', 'nn_tuned_2layer', 'nn_tuned_3layer', 'nn_tuned_4layer'],
    'All Features': ['lr_all', 'lasso_all', 'elasticnet_all', 'nn_all', 'nn_tuned_1layer_all', 'nn_tuned_2layer_all', 'nn_tuned_3layer_all', 'nn_tuned_4layer_all']
}

def safe_filename(name):
    """Create a filename-safe version of model names"""
    return name.lower().replace(' ', '_').replace('(', '').replace(')', '')

for group_name, model_keys in model_groups.items():
    # Get the prediction columns that exist
    pred_cols = [MODELS[k]['col'] for k in model_keys if k in MODELS and MODELS[k]['col'] in portfolios]
    
    if not pred_cols:
        print(f"Skipping {group_name} - no portfolios available")
        continue
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for pred_col in pred_cols:
        portfolio = portfolios[pred_col]
        
        # Calculate long-short returns
        ls_ret = portfolio['long_ret'].fillna(0) - portfolio['short_ret'].fillna(0)
        
        # Calculate log cumulative returns
        log_cum_ret = np.log(1 + ls_ret).cumsum()
        
        # Get model name for legend
        model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
        
        # Plot
        ax.plot(log_cum_ret.index, log_cum_ret.values, label=model_name, linewidth=1.5)
    
    ax.set_title(f'Log Cumulative Returns - {group_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Log Cumulative Return', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    
    plt.tight_layout()
    
    # Save the figure
    filename = f'log_cumulative_returns_{safe_filename(group_name)}.png'
    filepath = os.path.join(FIGURES, filename)
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"Saved: {filename}")
    
    plt.show()

In [5]:
# Print summary statistics
print("\nFinal Log Cumulative Returns (as of last date):")
print("=" * 80)
for pred_col in prediction_columns:
    if pred_col not in portfolios:
        continue
    portfolio = portfolios[pred_col]
    ls_ret = portfolio['long_ret'].fillna(0) - portfolio['short_ret'].fillna(0)
    log_cum_ret = np.log(1 + ls_ret).cumsum()
    final_value = log_cum_ret.iloc[-1] if len(log_cum_ret) > 0 else 0
    
    model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
    print(f"{model_name:45s}: {final_value:>8.4f} ({np.exp(final_value)-1:>7.2%})")


Final Log Cumulative Returns (as of last date):
Linear Regression                            :   0.2184 ( 24.40%)
LASSO                                        :   0.1580 ( 17.12%)
Elastic Net                                  :   0.1576 ( 17.07%)
Neural Network                               :  -0.1365 (-12.76%)
Neural Network Tuned (1-Layer)               :  -1.5193 (-78.11%)
Linear Regression (All Features)             :   1.3684 (292.91%)
LASSO (All Features)                         :   1.3295 (277.91%)
Elastic Net (All Features)                   :   1.3239 (275.81%)
Neural Network (All Features)                :   0.0249 (  2.52%)


# Time series tests (Fama-French regressions)

In [6]:
# Prepare the factor data
ff_factors_daily = ff_factors.copy() * 100  # Convert to percentage
ff_factors_daily['const'] = 1

# Use minimum 10 stocks portfolios as the main results
portfolios_to_test = portfolios_filtered[10]

# Calculate excess returns (long-short) for each portfolio and convert to percentage
portfolio_returns = {}
for pred_col in prediction_columns:
    if pred_col not in portfolios_to_test:
        continue
    portfolio = portfolios_to_test[pred_col]
    ls_ret = portfolio['long_ret'].fillna(0) - portfolio['short_ret'].fillna(0)
    portfolio_returns[pred_col] = ls_ret * 100  # Convert to percentage

# Merge all portfolio returns with factors
returns_df = pd.DataFrame(portfolio_returns)
returns_df.index.name = 'Date'

# Merge with factors
data = returns_df.merge(ff_factors_daily, left_index=True, right_index=True, how='inner')

print(f"Prepared regression data with {len(data)} observations")

Prepared regression data with 2768 observations


In [7]:
print("Running time series regressions...")
print("=" * 80)

# Dictionary to store all regression results
regression_results = {
    'CAPM': [],
    'FF3': [],
    'FF5': [],
    'FF6': []
}

# Run regressions for each prediction column
cols_to_test = [c for c in prediction_columns if c in data.columns]

for pred_col in cols_to_test:
    y = data[pred_col]
    
    # CAPM: alpha + beta * (Mkt-RF) + RF
    X_capm = data[['const', 'Mkt-RF']]
    model_capm = sm.OLS(y, X_capm).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['CAPM'].append(model_capm)
    
    # FF3: alpha + Mkt-RF + SMB + HML
    X_ff3 = data[['const', 'Mkt-RF', 'SMB', 'HML']]
    model_ff3 = sm.OLS(y, X_ff3).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['FF3'].append(model_ff3)
    
    # FF5: alpha + Mkt-RF + SMB + HML + RMW + CMA
    X_ff5 = data[['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
    model_ff5 = sm.OLS(y, X_ff5).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['FF5'].append(model_ff5)
    
    # FF6: FF5 + Mom
    X_ff6 = data[['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']]
    model_ff6 = sm.OLS(y, X_ff6).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['FF6'].append(model_ff6)

print("Regressions complete!")
print(f"Total regressions run: {len(regression_results) * len(cols_to_test)}")

Running time series regressions...
Regressions complete!
Total regressions run: 36


In [8]:
# Create LaTeX tables for each factor model

# Define variable names for better display
var_names = {
    'const': 'Alpha',
    'Mkt-RF': 'Mkt-RF',
    'SMB': 'SMB',
    'HML': 'HML',
    'RMW': 'RMW',
    'CMA': 'CMA',
    'Mom': 'Mom'
}

# Column names from MODELS dict
cols_to_test = [c for c in prediction_columns if c in data.columns]
col_names_short = [next((m['name'] for m in MODELS.values() if m['col'] == c), c) for c in cols_to_test]

print("\n" + "=" * 80)
print("GENERATING LATEX TABLES (Returns already in percentage)")
print("=" * 80)

# Store tables for LaTeX export
latex_tables = {}

# CAPM Table
print("\n1. CAPM Model")
print("-" * 80)
tbl_capm = linear_regression(regression_results['CAPM'])
tbl_capm.set_var_list(['const', 'Mkt-RF'])
tbl_capm.rename_variables(var_names)
tbl_capm.rename_columns(col_names_short)
tbl_capm.aux_stat = 't'
tbl_capm.render(R2=True, obs=True)
latex_tables['CAPM'] = tbl_capm
print(tbl_capm.tbl.to_string())

# FF3 Table
print("\n2. Fama-French 3-Factor Model")
print("-" * 80)
tbl_ff3 = linear_regression(regression_results['FF3'])
tbl_ff3.set_var_list(['const', 'Mkt-RF', 'SMB', 'HML'])
tbl_ff3.rename_variables(var_names)
tbl_ff3.rename_columns(col_names_short)
tbl_ff3.aux_stat = 't'
tbl_ff3.render(R2=True, obs=True)
latex_tables['FF3'] = tbl_ff3
print(tbl_ff3.tbl.to_string())

# FF5 Table
print("\n3. Fama-French 5-Factor Model")
print("-" * 80)
tbl_ff5 = linear_regression(regression_results['FF5'])
tbl_ff5.set_var_list(['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA'])
tbl_ff5.rename_variables(var_names)
tbl_ff5.rename_columns(col_names_short)
tbl_ff5.aux_stat = 't'
tbl_ff5.render(R2=True, obs=True)
latex_tables['FF5'] = tbl_ff5
print(tbl_ff5.tbl.to_string())

# FF6 Table
print("\n4. Fama-French 6-Factor Model (FF5 + Momentum)")
print("-" * 80)
tbl_ff6 = linear_regression(regression_results['FF6'])
tbl_ff6.set_var_list(['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom'])
tbl_ff6.rename_variables(var_names)
tbl_ff6.rename_columns(col_names_short)
tbl_ff6.aux_stat = 't'
tbl_ff6.render(R2=True, obs=True)
latex_tables['FF6'] = tbl_ff6
print(tbl_ff6.tbl.to_string())


GENERATING LATEX TABLES (Returns already in percentage)

1. CAPM Model
--------------------------------------------------------------------------------
         Linear Regression           LASSO     Elastic Net  Neural Network Neural Network Tuned (1-Layer) Linear Regression (All Features) LASSO (All Features) Elastic Net (All Features) Neural Network (All Features)
Alpha        0.08THREESTAR   0.07THREESTAR   0.07THREESTAR            0.00                          -0.02                    0.07THREESTAR        0.07THREESTAR              0.07THREESTAR                          0.01
                    (4.20)          (3.45)          (3.45)          (0.12)                        (-0.87)                           (3.72)               (3.99)                     (3.98)                        (1.29)
Mkt-RF      -1.14THREESTAR  -0.98THREESTAR  -0.98THREESTAR  -0.10THREESTAR                 -0.64THREESTAR                   -0.38THREESTAR       -0.29THREESTAR             -0.29THREESTAR          